In [ ]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split,RandomizedSearchCV
from xgboost import XGBClassifier
from sklearn.ensemble import RandomForestClassifier,ExtraTreesClassifier
from lightgbm import LGBMClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix
)

In [3]:
df_train=pd.read_csv(r"F:\playground-series-s6e6\train.csv")
df_test=pd.read_csv(r"F:\playground-series-s6e6\test.csv")

In [4]:
df_train.head()

,id,alpha,delta,u,g,r,i,z,redshift,spectral_type,galaxy_population,class
0,0,147.734256,16.959273,25.472123,21.895559,20.357926,19.257113,18.621057,0.408982,M,Red_Sequence,GALAXY
1,1,127.988677,32.346716,20.778509,19.087062,17.587208,17.226067,16.786433,0.157976,M,Red_Sequence,GALAXY
2,2,179.792648,35.344843,21.035203,21.079128,21.171840,20.582629,20.557366,2.823770,O/B,Blue_Cloud,QSO
3,3,225.818295,48.569421,23.305056,21.050736,19.017754,18.365658,17.914952,0.536099,M,Red_Sequence,GALAXY
4,4,141.836135,19.342852,21.703158,19.471680,18.234449,17.899447,17.616185,0.555761,M,Red_Sequence,GALAXY


In [5]:
df_test.head()

,id,alpha,delta,u,g,r,i,z,redshift,spectral_type,galaxy_population
0,577347,120.719779,23.924249,23.668066,21.951680,21.086183,20.180032,19.202124,0.429042,G/K,Red_Sequence
1,577348,219.414419,42.171651,24.902933,22.338822,20.732163,19.860330,19.687691,0.867305,M,Red_Sequence
2,577349,173.568731,-1.756400,19.427591,18.474633,17.551314,16.570674,16.176765,0.224234,G/K,Blue_Cloud
3,577350,184.903993,-1.411074,23.121029,21.526855,20.670159,20.417633,20.699095,0.066507,G/K,Red_Sequence
4,577351,222.487816,15.381403,25.094282,22.643981,21.123173,19.439500,19.094158,0.977218,M,Red_Sequence


In [6]:
X=df_train.drop(columns=["class"])
y=df_train["class"]
X_test=df_test.copy()

In [7]:
train_ids=X["id"]
test_ids=X_test["id"]

In [8]:
X=X.drop(columns=["id"])
X_test=X_test.drop(columns=["id"])

In [10]:
X.columns.to_list()

['alpha',
 'delta',
 'u',
 'g',
 'r',
 'i',
 'z',
 'redshift',
 'spectral_type',
 'galaxy_population']

In [11]:
X_test.columns.to_list()

['alpha',
 'delta',
 'u',
 'g',
 'r',
 'i',
 'z',
 'redshift',
 'spectral_type',
 'galaxy_population']

In [12]:
print(X.select_dtypes(include="object").columns)
print(X_test.select_dtypes(include="object").columns)

Index(['spectral_type', 'galaxy_population'], dtype='object')
Index(['spectral_type', 'galaxy_population'], dtype='object')


In [13]:
combined=pd.concat([X,X_test],axis=0)
combined=pd.get_dummies(
    combined,
    columns=["spectral_type", "galaxy_population"],
    drop_first=False
)

In [14]:
X=combined.iloc[:len(X)]
X_test=combined.iloc[len(X):]

In [15]:
X.columns.to_list()

['alpha',
 'delta',
 'u',
 'g',
 'r',
 'i',
 'z',
 'redshift',
 'spectral_type_A/F',
 'spectral_type_G/K',
 'spectral_type_M',
 'spectral_type_O/B',
 'galaxy_population_Blue_Cloud',
 'galaxy_population_Red_Sequence']

In [16]:
X_test.columns.to_list()

['alpha',
 'delta',
 'u',
 'g',
 'r',
 'i',
 'z',
 'redshift',
 'spectral_type_A/F',
 'spectral_type_G/K',
 'spectral_type_M',
 'spectral_type_O/B',
 'galaxy_population_Blue_Cloud',
 'galaxy_population_Red_Sequence']

In [ ]:
le=LabelEncoder()
y_encoded=le.fit_transform(y)
print(dict(zip(le.classes_,le.transform(le.classes_))))

{'GALAXY': 0, 'QSO': 1, 'STAR': 2}


In [21]:
X_train,X_val,y_train,y_val=train_test_split(X,y_encoded,test_size=0.2,random_state=42,stratify=y_encoded)

In [ ]:
xgb = XGBClassifier(
    objective="multi:softprob",
    eval_metric="mlogloss",
    random_state=42,
    n_jobs=-1
)

param_dist = {
    "n_estimators": [300, 500, 700, 1000],
    "learning_rate": [0.01, 0.03, 0.05, 0.1],
    "max_depth": [3, 4, 5, 6, 7, 8],
    "min_child_weight": [1, 3, 5, 7, 10],
    "subsample": [0.6, 0.7, 0.8, 0.9, 1.0],
    "colsample_bytree": [0.6, 0.7, 0.8, 0.9, 1.0],
    "gamma": [0, 0.1, 0.2, 0.3, 0.5],
    "reg_alpha": [0, 0.01, 0.1, 1],
    "reg_lambda": [0.1, 1, 2, 5, 10]
}

search = RandomizedSearchCV(
    estimator=xgb,
    param_distributions=param_dist,
    n_iter=50,      
    scoring="f1_weighted",
    cv=3,
    verbose=2,
    n_jobs=-1,
    random_state=42
)

search.fit(X_train, y_train)

print("Best Score:", search.best_score_)
print("Best Params:")
print(search.best_params_)

Fitting 3 folds for each of 50 candidates, totalling 150 fits
Best Score: 0.967102970042517
Best Params:
{'subsample': 0.9, 'reg_lambda': 2, 'reg_alpha': 0.01, 'n_estimators': 1000, 'min_child_weight': 7, 'max_depth': 7, 'learning_rate': 0.05, 'gamma': 0, 'colsample_bytree': 0.7}


In [61]:
best_model = XGBClassifier(
    **search.best_params_,
    objective="multi:softprob",
    eval_metric="mlogloss",
    random_state=42,
    n_jobs=-1
)

best_model.fit(X_train, y_train)
preds = best_model.predict(X_val)

print("Accuracy:", accuracy_score(y_val, preds))
print(classification_report(y_val, preds))

cm = confusion_matrix(y_val, preds)
print(cm)

Accuracy: 0.9669697756993159
              precision    recall  f1-score   support

           0       0.98      0.98      0.98     75496
           1       0.96      0.96      0.96     23429
           2       0.93      0.92      0.93     16545

    accuracy                           0.97    115470
   macro avg       0.96      0.95      0.96    115470
weighted avg       0.97      0.97      0.97    115470

[[73853   699   944]
 [  645 22535   249]
 [ 1147   130 15268]]


In [ ]:
rf=RandomForestClassifier(
    n_estimators=300,
    random_state=42,
    n_jobs=-1)

rf.fit(X_train, y_train)

rf_preds = rf.predict(X_val)

print("RF Accuracy:", accuracy_score(y_val, rf_preds))
print(classification_report(y_val, rf_preds))

RF Accuracy: 0.9598770243353252
              precision    recall  f1-score   support

           0       0.97      0.97      0.97     75496
           1       0.96      0.96      0.96     23429
           2       0.91      0.90      0.90     16545

    accuracy                           0.96    115470
   macro avg       0.95      0.94      0.95    115470
weighted avg       0.96      0.96      0.96    115470



In [62]:
cm_rf=confusion_matrix(y_val, rf_preds)
print(cm_rf)

[[73492   767  1237]
 [  663 22471   295]
 [ 1573    98 14874]]


In [ ]:
lgbm=LGBMClassifier(
    n_estimators=500,
    learning_rate=0.1,
    num_leaves=31,
    random_state=42)

lgbm.fit(X_train, y_train)

lgbm_preds = lgbm.predict(X_val)

print("LGBM Accuracy:", accuracy_score(y_val, lgbm_preds))
print(classification_report(y_val, lgbm_preds))

[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.002616 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 2052
[LightGBM] [Info] Number of data points in the train set: 461877, number of used features: 14
[LightGBM] [Info] Start training from score -0.424925
[LightGBM] [Info] Start training from score -1.595051
[LightGBM] [Info] Start training from score -1.942935
LGBM Accuracy: 0.9654369100199186
              precision    recall  f1-score   support

           0       0.98      0.98      0.98     75496
           1       0.96      0.96      0.96     23429
           2       0.92      0.92      0.92     16545

    accuracy                           0.97    115470
   macro avg       0.95      0.95      0.95    115470
weighted avg       0.97      0.97      0.97    115470



In [63]:
cm_lgbm = confusion_matrix(y_val, lgbm_preds)
print(cm_lgbm)

[[73704   769  1023]
 [  684 22518   227]
 [ 1180   108 15257]]


In [51]:
lr=LogisticRegression(
    max_iter=5000,
    n_jobs=-1,
    multi_class="multinomial",
    solver="saga"
)

lr.fit(X_train,y_train)

lr_preds = lr.predict(X_val)

print("LR Accuracy:",accuracy_score(y_val,lr_preds))
print(classification_report(y_val, lr_preds))

LR Accuracy: 0.9272451719061228
              precision    recall  f1-score   support

           0       0.95      0.96      0.95     75496
           1       0.93      0.92      0.92     23429
           2       0.83      0.81      0.82     16545

    accuracy                           0.93    115470
   macro avg       0.90      0.89      0.90    115470
weighted avg       0.93      0.93      0.93    115470



In [64]:
cm_lr = confusion_matrix(y_val, lr_preds)
print(cm_lr)

[[72196  1515  1785]
 [  949 21444  1036]
 [ 3028    88 13429]]


In [65]:
et = ExtraTreesClassifier(
    n_estimators=500,
    random_state=42,
    n_jobs=-1
)

et.fit(X_train,y_train)

et_preds = et.predict(X_val)

print("ET Accuracy:",accuracy_score(y_val,et_preds))
print(classification_report(y_val, et_preds))

ET Accuracy: 0.9561704338789296
              precision    recall  f1-score   support

           0       0.97      0.98      0.97     75496
           1       0.96      0.96      0.96     23429
           2       0.91      0.87      0.89     16545

    accuracy                           0.96    115470
   macro avg       0.94      0.93      0.94    115470
weighted avg       0.96      0.96      0.96    115470



In [66]:
cm_et=confusion_matrix(y_val, et_preds)
print(cm_et)

[[73647   789  1060]
 [  658 22394   377]
 [ 1997   180 14368]]
